In [244]:
import logging
import pandas as pd
import numpy as np
import os

logging.basicConfig(level=logging.INFO)

class FileValidationError(Exception):
    pass

class FileMissingValueError(FileValidationError):
    pass

def load_results(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileValidationError("File not found!!")  
    elif os.path.getsize(filepath) == 0:
        raise FileMissingValueError("File exists but contains no value or data")
    else:
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        logging.info(f"Loaded {filepath} — shape: {df.shape}")
        return df

try:
    al = load_results("data/raw/Allocated Limit for Honble MPs.csv")
    wc = load_results("data/raw/Works Completed.csv")
    ws = load_results("data/raw/Works Sanctioned.csv")

except FileValidationError as e:
    logging.error(f"Loading Failed: {e}")
    raise

INFO:root:Loaded data/raw/Allocated Limit for Honble MPs.csv — shape: (544, 5)
INFO:root:Loaded data/raw/Works Completed.csv — shape: (15001, 11)
INFO:root:Loaded data/raw/Works Sanctioned.csv — shape: (6001, 12)


In [245]:
import re

def clean_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    
    dfCols = list(df.columns)
    columns_cleaned = []

    for col in dfCols:
        cleaned_names = re.sub("[\.\'\()\₹]", '',str(col)).lower().strip()
        cleaned_names = cleaned_names.replace(" ", "_")
        columns_cleaned.append(cleaned_names)

    df.columns = (columns_cleaned)
    return df


clean_dataframe_columns(al).columns
clean_dataframe_columns(wc).columns
clean_dataframe_columns(ws).columns

Index(['sr_no', 'work_category', 'work', 'state', 'ida',
       'honble_members_of_parliament', 'constituency', 'work_description',
       'recommended_date', 'sanction_date', 'sanction_amount', 'work_status'],
      dtype='object')

In [246]:
al.columns, ws.columns, wc.columns

(Index(['sr_no', 'state', 'honble_members_of_parliaments', 'constituency',
        'allocated_amount'],
       dtype='object'),
 Index(['sr_no', 'work_category', 'work', 'state', 'ida',
        'honble_members_of_parliament', 'constituency', 'work_description',
        'recommended_date', 'sanction_date', 'sanction_amount', 'work_status'],
       dtype='object'),
 Index(['sr_no', 'work_category', 'work', 'state', 'ida', 'work_description',
        'honble_members_of_parliament', 'constituency', 'image',
        'completion_date', 'amount_disbursed'],
       dtype='object'))

In [247]:
def diagnose_dataframe(df: pd.DataFrame, name: str):
    logging.info(f"\n Dataframe -> {name}\n")
    logging.info(f" Shape: \n{df.shape}\n")
    logging.info(f" Dataframe Datatypes \n{df.dtypes}\n")
    logging.info(f" Dataframe null columns \n{df.isna().sum()[df.isna().sum()>0]}\n")
    logging.info(f" Datarframe duplicates \n{df.duplicated().sum()}")

    
diagnose_dataframe(al, "Allocated Limit for Honble MPs")
diagnose_dataframe(wc, "Works Completed")
diagnose_dataframe(ws, "Works Sanctioned")

INFO:root:
 Dataframe -> Allocated Limit for Honble MPs

INFO:root: Shape: 
(544, 5)

INFO:root: Dataframe Datatypes 
sr_no                            object
state                            object
honble_members_of_parliaments    object
constituency                     object
allocated_amount                 object
dtype: object

INFO:root: Dataframe null columns 
allocated_amount    1
dtype: int64

INFO:root: Datarframe duplicates 
0
INFO:root:
 Dataframe -> Works Completed

INFO:root: Shape: 
(15001, 11)

INFO:root: Dataframe Datatypes 
sr_no                           object
work_category                   object
work                            object
state                           object
ida                             object
work_description                object
honble_members_of_parliament    object
constituency                    object
image                           object
completion_date                 object
amount_disbursed                object
dtype: object

INFO:root:

In [248]:
print(al.isna().sum()[al.isna().sum()>0])
print(wc.isna().sum()[wc.isna().sum()>0])
print(ws.isna().sum()[ws.isna().sum()>0])

allocated_amount    1
dtype: int64
work_description      55
image               5054
amount_disbursed       6
dtype: int64
work_description    38
dtype: int64


In [249]:
def numeric_conversion(df:pd.DataFrame, columns: list) -> pd.DataFrame:
    for cols in columns:
        df[cols] = pd.to_numeric(df[cols], errors="coerce", downcast="float")
    return df

al = numeric_conversion(al, ["allocated_amount"])
wc = numeric_conversion(wc, ["amount_disbursed"])
ws = numeric_conversion(ws, ["sanction_amount"])

In [250]:
print(al.isna().sum()[al.isna().sum()>0])
print(wc.isna().sum()[wc.isna().sum()>0])
print(ws.isna().sum()[ws.isna().sum()>0])

allocated_amount    1
dtype: int64
work_description      55
image               5054
amount_disbursed       6
dtype: int64
work_description    38
dtype: int64


In [251]:
def replace_strings_with_real_nulls(df: pd.DataFrame) -> pd.DataFrame:
    df = df.replace("NaN", np.nan)
    return df
    
al = replace_strings_with_real_nulls(al)
wc = replace_strings_with_real_nulls(wc)
ws = replace_strings_with_real_nulls(ws)

In [252]:
def strip_whitespace(df: pd.DataFrame) -> pd.DataFrame:
    for col_name in df.columns:
        if df[col_name].dtype == "object":
            df[col_name] = df[col_name].str.strip()
            df[col_name] = df[col_name].str.replace(r"\t", " ", regex=True) 

    return df

al = strip_whitespace(al)
wc = strip_whitespace(wc)
ws = strip_whitespace(ws)

# al.style.format({"allocated_amount": "{:,.2f}"})
# wc.style.format({"amount_disbursed": "{:,.2f}"})
# wc.style.format({"sanction_amount": "{:,.2f}"})
# DO-NOT UNCOMMENT THESE!!

In [264]:
def convert_real_datetime(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    for cols in columns:
        initial_count =  df[cols].isna().sum()
        df[cols] = pd.to_datetime(df[cols], errors="coerce",  format="%d-%b-%Y")
        failed_count = df[cols].isna().sum()
        failed = failed_count - initial_count
        logging.info(f"Column '{cols}': {failed} unparseable entries turned into NaT.")
    return df

wc_datetime_list = ["completion_date"]
ws_datetime_list = ["recommended_date", "sanction_date"]
wc = convert_real_datetime(wc, wc_datetime_list)
ws = convert_real_datetime(ws, ws_datetime_list)

INFO:root:Column 'completion_date': 0 unparseable entries turned into NaT.
INFO:root:Column 'recommended_date': 0 unparseable entries turned into NaT.
INFO:root:Column 'sanction_date': 0 unparseable entries turned into NaT.


In [265]:
def drop_duplicate_rows(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = df.shape[0]
    df = df.drop_duplicates()
    after = df.shape[0]
    logging.info(f"{name}: dropped {before - after} duplicate rows")
    return df

al = drop_duplicate_rows(al, "Allocated Limit")
ws = drop_duplicate_rows(ws, "Works Sanctioned")
wc = drop_duplicate_rows(wc, "Works Completed")

INFO:root:Allocated Limit: dropped 0 duplicate rows
INFO:root:Works Sanctioned: dropped 0 duplicate rows
INFO:root:Works Completed: dropped 0 duplicate rows


In [266]:
al = al.drop(columns=["sr_no"])
ws = ws.drop(columns=["sr_no"])
wc = wc.drop(columns=["sr_no", "image"])

In [267]:
diagnose_dataframe(al, "Allocated Limit (cleaned)")
diagnose_dataframe(ws, "Works Sanctioned (cleaned)")
diagnose_dataframe(wc, "Works Completed (cleaned)")

INFO:root:
 Dataframe -> Allocated Limit (cleaned)

INFO:root: Shape: 
(543, 4)

INFO:root: Dataframe Datatypes 
state                             object
honble_members_of_parliaments     object
constituency                      object
allocated_amount                 float64
dtype: object

INFO:root: Dataframe null columns 
allocated_amount    1
dtype: int64

INFO:root: Datarframe duplicates 
0
INFO:root:
 Dataframe -> Works Sanctioned (cleaned)

INFO:root: Shape: 
(6000, 11)

INFO:root: Dataframe Datatypes 
work_category                           object
work                                    object
state                                   object
ida                                     object
honble_members_of_parliament            object
constituency                            object
work_description                        object
recommended_date                datetime64[ns]
sanction_date                   datetime64[ns]
sanction_amount                        float64
work_status    